In [ ]:
# ============================================================
# ÉTAPE 5 : PRÉDICTION DE MATCHS ATP — VERSION FINALE
# ============================================================
#
# Deux modes :
#   1. predict_match()   — prédiction manuelle (2 noms + date + contexte)
#   2. predict_ongoing() — prédiction sur tous les matchs ongoing
#
# Nouveautés vs version précédente :
#   ✅ match_date obligatoire → fatigue calculée à la bonne date
#   ✅ Nouvelles features (rank_ratio, momentum, serve_dom...)
#   ✅ get_player_stats() avec date pour éviter le leakage
#   ✅ FEATURE_COLS aligné avec le modèle entraîné
# ============================================================

import pandas as pd
import numpy as np
import joblib
import warnings
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

# ─────────────────────────────────────────────────────────────
# A. CHARGEMENT
# ─────────────────────────────────────────────────────────────

# Redéfinition de PipelineWrapper (nécessaire si le modèle est un RF wrappé)
from sklearn.pipeline import Pipeline

class PipelineWrapper:
    def __init__(self, pipeline):
        self.pipeline = pipeline
    def predict(self, X):
        return self.pipeline.predict(X)
    def predict_proba(self, X):
        return self.pipeline.predict_proba(X)

# ── Chargement du meilleur modèle ──
model_data = joblib.load("../../models/tennis/best_model.pkl")
model      = model_data["model"]
model_name = model_data["model_name"]

# ── Chargement des données ──
feat_df  = pd.read_csv("../../data/tennis/atp_features_clean_final.csv")
df_orig  = pd.read_csv("../../data/tennis/atp_clean.csv", parse_dates=["tourney_date"])
df_orig  = df_orig.sort_values("tourney_date").reset_index(drop=True)
ongoing  = pd.read_csv("../../data/tennis/ongoing_tourneys.csv")

# ── Préparation de feat_clean ──
feat_clean = feat_df.copy()
feat_clean = feat_clean.sort_values("year").reset_index(drop=True)

# Ajout de la colonne p1_name pour la recherche par nom
# (doit correspondre à la façon dont elle a été créée dans le feature engineering)
print("Construction index joueur → feat_clean...")
if "p1_name" not in feat_clean.columns:
    # Fallback si p1_name absent du CSV
    player_names_feat = []
    for i, row_orig in df_orig.iterrows():
        player_names_feat.append(row_orig["winner_name"])
        player_names_feat.append(row_orig["loser_name"])
    feat_clean["player_name"] = player_names_feat
    USE_PLAYER_NAME = False
else:
    USE_PLAYER_NAME = True
    print("   → Utilisation de p1_name (colonne existante)")

print(f"✅ Index construit : {len(feat_clean)} lignes")

# ── FEATURE_COLS : liste des features attendues par le modèle ──
# Exclure les colonnes non-features
FEATURE_COLS = [c for c in feat_clean.columns
                if c not in ["label", "year", "diff_last20_won",
                             "p1_name", "p2_name", "player_name"]]

# Ajouter les nouvelles features dérivées créées lors de l'entraînement optimisé
# (elles ne sont pas dans feat_clean mais dans le modèle entraîné)
NEW_FEATURES = [
    "rank_ratio",
    "p1_momentum", "p2_momentum", "diff_momentum",
    "p1_serve_dom", "p2_serve_dom", "diff_serve_dom",
    "diff_bp_pressure",
]
# On les ajoute seulement si le modèle les connaît
if hasattr(model_data, "get") and "feature_names" in model_data:
    model_features = model_data["feature_names"]
    FEATURE_COLS = model_features  # utiliser exactement les features du modèle
else:
    for f in NEW_FEATURES:
        if f not in FEATURE_COLS:
            FEATURE_COLS.append(f)

N_MATCHES = len(df_orig)

print(f"✅ Modèle chargé    : {model_name}")
print(f"   Features         : {len(FEATURE_COLS)}")
print(f"   Matchs en base   : {N_MATCHES:,}")
print(f"   Matchs ongoing   : {len(ongoing)}")

# ─────────────────────────────────────────────────────────────
# B. ENCODAGES CONTEXTUELS
# ─────────────────────────────────────────────────────────────

ROUND_ORDER = {"R128":1,"R64":2,"R32":3,"R16":4,"QF":5,"SF":6,"F":7,"RR":3,"BR":6}
LEVEL_ORDER = {"G":5,"M":4,"F":4,"A":3,"D":2,"C":1}
HAND_MAP    = {"R":1,"L":-1,"U":0}
INDOOR_MAP  = {"Y":1,"N":0,"Unknown":0}

# ─────────────────────────────────────────────────────────────
# C. FONCTION : RÉCUPÉRER LES STATS D'UN JOUEUR
# ─────────────────────────────────────────────────────────────

def get_player_stats(player_name: str, match_date: pd.Timestamp) -> dict:
    """
    Récupère les dernières stats rolling d'un joueur AVANT une date donnée.

    Paramètres :
        player_name : nom exact du joueur
        match_date  : date du match à prédire (pd.Timestamp)
                      → seuls les matchs AVANT cette date sont utilisés

    Retourne :
        dict avec toutes les stats p1_* du joueur + infos de debug
    """
    mask    = (df_orig["winner_name"] == player_name) | \
              (df_orig["loser_name"]  == player_name)

    # Uniquement les matchs AVANT la date du match à prédire (no leakage)
    matches = df_orig[mask & (df_orig["tourney_date"] < match_date)] \
                .sort_values("tourney_date")

    if len(matches) == 0:
        return None

    last          = matches.iloc[-1]
    is_winner     = last["winner_name"] == player_name
    label_val     = 1 if is_winner else 0
    expected_rank = float(last["winner_rank"] if is_winner else last["loser_rank"])

    # ── Recherche de la ligne dans feat_clean ──
    if USE_PLAYER_NAME:
        candidates = feat_clean[feat_clean["p1_name"] == player_name]
    else:
        candidates = feat_clean[feat_clean["player_name"] == player_name]

    candidates = candidates[candidates["label"] == label_val]
    exact      = candidates[candidates["p1_rank"] == expected_rank]
    feat_row   = exact.iloc[-1] if len(exact) > 0 else candidates.iloc[-1]

    # Extraire toutes les stats p1_*
    stats = {col.replace("p1_", ""): feat_row[col]
             for col in feat_clean.columns if col.startswith("p1_")
             and col not in ["p1_name"]}

    # ── Fatigue RÉELLE calculée par rapport à la date du match ──
    # Nombre de matchs dans les 14 jours avant la date du match
    fatigue_window = matches[
        matches["tourney_date"] >= match_date - pd.Timedelta(days=14)
    ]
    stats["fatigue_14d"]     = len(fatigue_window)
    stats["days_since_last"] = (match_date - last["tourney_date"]).days

    # Infos de debug
    stats["_last_match_date"] = last["tourney_date"].date()
    stats["_last_opponent"]   = last["loser_name"] if is_winner else last["winner_name"]
    stats["_last_result"]     = "Victoire" if is_winner else "Défaite"

    return stats


# ─────────────────────────────────────────────────────────────
# D. FONCTION : CONSTRUIRE LES FEATURES D'UN MATCH
# ─────────────────────────────────────────────────────────────

def build_match_features(p1_stats: dict, p2_stats: dict,
                          surface: str, tourney_level: str,
                          round_str: str, best_of: int,
                          indoor: str, month: int,
                          h2h_n: int = 0,
                          h2h_win_rate_p1: float = np.nan,
                          h2h_win_rate_surf: float = np.nan) -> pd.DataFrame:
    """
    Construit la ligne de features complète pour un match donné.
    Inclut les nouvelles features dérivées de l'entraînement optimisé.
    """
    row = {}

    # ── Features statiques ──
    row["p1_rank"] = p1_stats.get("rank", 999)
    row["p1_age"]  = p1_stats.get("age",  np.nan)
    row["p1_ht"]   = p1_stats.get("ht",   np.nan)
    row["p2_rank"] = p2_stats.get("rank", 999)
    row["p2_age"]  = p2_stats.get("age",  np.nan)
    row["p2_ht"]   = p2_stats.get("ht",   np.nan)

    # ── Différentiels statiques ──
    row["diff_rank"]     = row["p1_rank"]    - row["p2_rank"]
    row["diff_rank_pts"] = p1_stats.get("rank_pts", 0) - p2_stats.get("rank_pts", 0)
    row["diff_seed"]     = p1_stats.get("seed", 0)     - p2_stats.get("seed", 0)
    row["diff_age"]      = row["p1_age"]     - row["p2_age"]
    row["diff_ht"]       = row["p1_ht"]      - row["p2_ht"]
    row["same_hand"]     = int(p1_stats.get("hand", 0) == p2_stats.get("hand", 0))

    # ── Rolling features P1 et P2 ──
    rolling_stats = [
        "last5_won", "last10_won", "last20_won",
        "last10_ace_rate", "last10_df_rate", "last10_fs_in_pct",
        "last10_fs_won_pct", "last10_ss_won_pct", "last10_bp_saved_pct",
        "last10_n_matches",
        "last5_win_rate_Hard", "last5_win_rate_Clay",
        "last10_win_rate_Hard", "last10_win_rate_Clay",
        "last20_win_rate_Hard", "last20_win_rate_Clay",
        "fatigue_14d", "days_since_last",
    ]
    for stat in rolling_stats:
        row[f"p1_{stat}"] = p1_stats.get(stat, np.nan)
        row[f"p2_{stat}"] = p2_stats.get(stat, np.nan)

    # ── Différentiels rolling ──
    diff_stats = [
        "last5_won", "last5_ace_rate", "last5_df_rate", "last5_fs_in_pct",
        "last5_fs_won_pct", "last5_ss_won_pct", "last5_bp_saved_pct",
        "last10_won", "last10_ace_rate", "last10_df_rate", "last10_fs_in_pct",
        "last10_fs_won_pct", "last10_ss_won_pct", "last10_bp_saved_pct",
        "last10_n_matches",
        "last20_ace_rate", "last20_df_rate", "last20_fs_in_pct",
        "last20_fs_won_pct", "last20_ss_won_pct", "last20_bp_saved_pct",
        "last5_win_rate_Hard", "last5_win_rate_Clay",
        "last10_win_rate_Hard", "last10_win_rate_Clay",
        "last20_win_rate_Hard", "last20_win_rate_Clay",
        "fatigue_14d", "days_since_last",
    ]
    for stat in diff_stats:
        v1 = p1_stats.get(stat, np.nan)
        v2 = p2_stats.get(stat, np.nan)
        try:
            row[f"diff_{stat}"] = v1 - v2 if (not np.isnan(float(v1)) and
                                               not np.isnan(float(v2))) else np.nan
        except (TypeError, ValueError):
            row[f"diff_{stat}"] = np.nan

    # ── H2H ──
    row["h2h_n"]             = h2h_n
    row["h2h_win_rate_p1"]   = h2h_win_rate_p1
    row["h2h_win_rate_surf"] = h2h_win_rate_surf

    # ── Contexte match ──
    row["round_num"]         = ROUND_ORDER.get(round_str, 3)
    row["tourney_level_num"] = LEVEL_ORDER.get(tourney_level, 3)
    row["indoor_enc"]        = INDOOR_MAP.get(str(indoor), 0)
    row["month"]             = month
    row["best_of"]           = best_of
    row["surf_Clay"]         = int(surface == "Clay")
    row["surf_Grass"]        = int(surface == "Grass")
    row["surf_Hard"]         = int(surface == "Hard")

    # ── Nouvelles features dérivées (entraînement optimisé) ──
    # Ces features ont été ajoutées lors de l'étape d'optimisation
    # et doivent être recalculées ici pour correspondre au modèle

    # 1. Ratio de classement
    r1 = row["p1_rank"]
    r2 = row["p2_rank"]
    row["rank_ratio"] = float(np.clip(r1 / r2, 0.01, 100)) if r2 > 0 else np.nan

    # 2. Momentum (tendance récente vs long terme)
    p1_l5  = row.get("p1_last5_won",  np.nan)
    p1_l20 = row.get("p1_last20_won", np.nan)
    p2_l5  = row.get("p2_last5_won",  np.nan)
    p2_l20 = row.get("p2_last20_won", np.nan)

    try:
        row["p1_momentum"]   = float(p1_l5) - float(p1_l20)
    except (TypeError, ValueError):
        row["p1_momentum"]   = np.nan
    try:
        row["p2_momentum"]   = float(p2_l5) - float(p2_l20)
    except (TypeError, ValueError):
        row["p2_momentum"]   = np.nan
    try:
        row["diff_momentum"] = row["p1_momentum"] - row["p2_momentum"]
    except (TypeError, ValueError):
        row["diff_momentum"] = np.nan

    # 3. Service dominance (ace_rate + fs_won_pct - df_rate)
    def serve_dom(stats, prefix):
        a = stats.get(f"{prefix}last10_ace_rate",    np.nan)
        f = stats.get(f"{prefix}last10_fs_won_pct",  np.nan)
        d = stats.get(f"{prefix}last10_df_rate",     np.nan)
        try:
            return float(a) + float(f) - float(d)
        except (TypeError, ValueError):
            return np.nan

    row["p1_serve_dom"]   = serve_dom(p1_stats, "")
    row["p2_serve_dom"]   = serve_dom(p2_stats, "")
    try:
        row["diff_serve_dom"] = row["p1_serve_dom"] - row["p2_serve_dom"]
    except (TypeError, ValueError):
        row["diff_serve_dom"] = np.nan

    # 4. Pression sous break
    bp  = row.get("diff_last10_bp_saved_pct", np.nan)
    fsw = row.get("diff_last10_fs_won_pct",   np.nan)
    try:
        row["diff_bp_pressure"] = float(bp) - float(fsw)
    except (TypeError, ValueError):
        row["diff_bp_pressure"] = np.nan

    # ── Alignement strict sur les features du modèle ──
    X = pd.DataFrame([row])
    # Ajouter les colonnes manquantes avec NaN
    for col in FEATURE_COLS:
        if col not in X.columns:
            X[col] = np.nan
    # Sélectionner uniquement les features dans le bon ordre
    X = X[FEATURE_COLS]
    return X


# ─────────────────────────────────────────────────────────────
# E. FONCTION : H2H HISTORIQUE
# ─────────────────────────────────────────────────────────────

def get_h2h(p1_name: str, p2_name: str, surface: str = None) -> tuple:
    """
    Calcule le H2H historique entre deux joueurs.
    Retourne (n_matchs, win_rate_p1_global, win_rate_p1_surface).
    """
    mask = (
        ((df_orig["winner_name"] == p1_name) & (df_orig["loser_name"] == p2_name)) |
        ((df_orig["winner_name"] == p2_name) & (df_orig["loser_name"] == p1_name))
    )
    h2h = df_orig[mask]
    n   = len(h2h)

    if n == 0:
        return 0, np.nan, np.nan

    p1_wins  = (h2h["winner_name"] == p1_name).sum()
    win_rate = p1_wins / n

    if surface:
        h2h_s     = h2h[h2h["surface"] == surface]
        n_s       = len(h2h_s)
        p1_wins_s = (h2h_s["winner_name"] == p1_name).sum()
        win_rate_s = p1_wins_s / n_s if n_s > 0 else np.nan
    else:
        win_rate_s = np.nan

    return n, win_rate, win_rate_s


# ─────────────────────────────────────────────────────────────
# F. PRÉDICTION SIMPLE — deux joueurs + date + contexte
# ─────────────────────────────────────────────────────────────

def predict_match(player1: str, player2: str,
                  match_date: str,
                  surface: str       = "Hard",
                  tourney_level: str = "M",
                  round_str: str     = "R32",
                  best_of: int       = 3,
                  indoor: str        = "N"):
    """
    Prédit le vainqueur d'un match entre deux joueurs.

    Paramètres :
        player1, player2  : noms exacts des joueurs (casse importante)
        match_date        : date du match "YYYY-MM-DD" (obligatoire)
        surface           : "Hard", "Clay", "Grass"
        tourney_level     : "G" (GC), "M" (Masters), "A" (500/250),
                            "D" (Challenger), "C" (ITF)
        round_str         : "R128","R64","R32","R16","QF","SF","F"
        best_of           : 3 ou 5 (5 uniquement en Grand Chelem)
        indoor            : "Y" ou "N"

    Exemple :
        predict_match("Jannik Sinner", "Carlos Alcaraz",
                      match_date="2026-05-05",
                      surface="Clay", tourney_level="M",
                      round_str="SF", best_of=3)
    """
    date  = pd.Timestamp(match_date)
    month = date.month

    # ── Récupération des stats ──
    p1_stats = get_player_stats(player1, date)
    p2_stats = get_player_stats(player2, date)

    if p1_stats is None:
        print(f"❌ Joueur introuvable dans le dataset : {player1}")
        return None
    if p2_stats is None:
        print(f"❌ Joueur introuvable dans le dataset : {player2}")
        return None

    # ── H2H ──
    h2h_n, h2h_wr, h2h_wr_s = get_h2h(player1, player2, surface)

    # ── Construction des features ──
    X = build_match_features(
        p1_stats, p2_stats,
        surface=surface, tourney_level=tourney_level,
        round_str=round_str, best_of=best_of,
        indoor=indoor, month=month,
        h2h_n=h2h_n, h2h_win_rate_p1=h2h_wr,
        h2h_win_rate_surf=h2h_wr_s
    )

    # ── Prédiction ──
    proba       = model.predict_proba(X)[0]
    p1_win_prob = proba[1]
    p2_win_prob = proba[0]
    winner      = player1 if p1_win_prob > 0.5 else player2
    confidence  = max(p1_win_prob, p2_win_prob)

    # ── Affichage ──
    bar_len = 40
    p1_bar  = int(p1_win_prob * bar_len)
    p2_bar  = bar_len - p1_bar

    print(f"\n{'='*55}")
    print(f"  🎾 {player1}")
    print(f"     vs")
    print(f"  🎾 {player2}")
    print(f"  📍 {surface} | {tourney_level} | {round_str} | Best of {best_of} | {match_date}")
    print(f"{'='*55}")

    print(f"\n  Derniers matchs connus :")
    print(f"  {player1[:25]:25} → {p1_stats['_last_match_date']} "
          f"({p1_stats['_last_result']} vs {p1_stats['_last_opponent'][:20]})")
    print(f"  {player2[:25]:25} → {p2_stats['_last_match_date']} "
          f"({p2_stats['_last_result']} vs {p2_stats['_last_opponent'][:20]})")

    print(f"\n  Fatigue (14j avant le match) :")
    print(f"  {player1[:25]:25} → {int(p1_stats['fatigue_14d'])} matchs | "
          f"repos : {int(p1_stats['days_since_last'])} jours")
    print(f"  {player2[:25]:25} → {int(p2_stats['fatigue_14d'])} matchs | "
          f"repos : {int(p2_stats['days_since_last'])} jours")

    if h2h_n > 0:
        p1_h2h_wins = int(h2h_wr * h2h_n)
        print(f"\n  H2H : {player1} mène {p1_h2h_wins}-{h2h_n - p1_h2h_wins} "
              f"({h2h_wr:.0%} de victoires)")
        if not np.isnan(h2h_wr_s):
            print(f"        Sur {surface} : {h2h_wr_s:.0%}")
    else:
        print(f"\n  H2H : Aucun affrontement précédent")

    print(f"\n  ┌{'─'*bar_len}┐")
    print(f"  │{'█'*p1_bar}{' '*p2_bar}│")
    print(f"  └{'─'*bar_len}┘")
    print(f"  {player1[:22]:22} {p1_win_prob:.1%}  │  {p2_win_prob:.1%} {player2[:22]}")
    print(f"\n  🏆 Vainqueur prédit : {winner}")
    print(f"  📊 Confiance        : {confidence:.1%}")
    print(f"  🤖 Modèle           : {model_name}")
    print(f"{'='*55}\n")

    return {
        "player1"          : player1,
        "player2"          : player2,
        "p1_prob"          : p1_win_prob,
        "p2_prob"          : p2_win_prob,
        "predicted_winner" : winner,
        "confidence"       : confidence,
    }


# ─────────────────────────────────────────────────────────────
# G. PRÉDICTION SUR LES MATCHS ONGOING
# ─────────────────────────────────────────────────────────────

def predict_ongoing(show_top: int = 30):
    """
    Prédit le vainqueur de tous les matchs dans ongoing_tourneys.csv.
    Le swap aléatoire P1/P2 garantit une évaluation honnête
    (le modèle ne sait pas qui est le vrai gagnant avant de prédire).
    La date réelle du match est utilisée pour calculer la fatigue.
    """
    import random
    random.seed(42)

    print(f"\n{'='*70}")
    print(f"  PRÉDICTIONS — {ongoing['tourney_name'].iloc[0]} ({len(ongoing)} matchs)")
    print(f"{'='*70}")

    preds  = []
    errors = []

    for _, match in ongoing.iterrows():

        # ── Date réelle du match ──
        match_date = pd.to_datetime(str(match["tourney_date"]), format="%Y%m%d")
        month      = match_date.month

        # ── Swap aléatoire P1/P2 ──
        if random.random() > 0.5:
            p1_name = match["winner_name"]
            p2_name = match["loser_name"]
        else:
            p1_name = match["loser_name"]
            p2_name = match["winner_name"]

        real_winner = match["winner_name"]
        surface     = match["surface"]
        level       = match["tourney_level"]
        round_s     = match["round"]
        best_of     = match["best_of"]
        indoor      = match.get("indoor", "N")

        # ── Stats joueurs à la date du match ──
        p1_stats = get_player_stats(p1_name, match_date)
        p2_stats = get_player_stats(p2_name, match_date)

        if p1_stats is None or p2_stats is None:
            errors.append(p1_name if p1_stats is None else p2_name)
            continue

        # ── H2H ──
        h2h_n, h2h_wr, h2h_wr_s = get_h2h(p1_name, p2_name, surface)

        # Retourner le H2H si P1 est le loser dans la base
        if p1_name == match["loser_name"] and not np.isnan(h2h_wr):
            h2h_wr   = 1 - h2h_wr
            h2h_wr_s = 1 - h2h_wr_s if not np.isnan(h2h_wr_s) else np.nan

        # ── Features + prédiction ──
        X = build_match_features(
            p1_stats, p2_stats,
            surface=surface, tourney_level=level,
            round_str=round_s, best_of=best_of,
            indoor=str(indoor), month=month,
            h2h_n=h2h_n, h2h_win_rate_p1=h2h_wr,
            h2h_win_rate_surf=h2h_wr_s
        )

        proba       = model.predict_proba(X)[0]
        p1_prob     = proba[1]
        pred_winner = p1_name if p1_prob > 0.5 else p2_name
        correct     = pred_winner == real_winner

        preds.append({
            "Round"       : round_s,
            "Joueur 1"    : p1_name,
            "Joueur 2"    : p2_name,
            "P(J1 gagne)" : f"{p1_prob:.1%}",
            "Prédit"      : pred_winner,
            "Réel"        : real_winner,
            "✅/❌"       : "✅" if correct else "❌",
            "_correct"    : correct,
            "_p1_prob"    : p1_prob,
        })

    results_df = pd.DataFrame(preds)
    accuracy   = results_df["_correct"].mean()
    n_correct  = results_df["_correct"].sum()

    # ── Affichage ──
    display_cols = ["Round","Joueur 1","Joueur 2","P(J1 gagne)","Prédit","Réel","✅/❌"]
    print(results_df[display_cols].head(show_top).to_string(index=False))

    if len(results_df) > show_top:
        print(f"  ... ({len(results_df) - show_top} matchs supplémentaires)")

    print(f"\n{'─'*70}")
    print(f"  📊 Résultats sur {len(results_df)} matchs :")
    print(f"     ✅ Bonnes prédictions : {n_correct} ({accuracy:.1%})")
    print(f"     ❌ Mauvaises          : {len(results_df)-n_correct} ({1-accuracy:.1%})")

    print(f"\n  📋 Précision par round :")
    round_acc = results_df.groupby("Round")["_correct"].agg(["sum","count","mean"])
    round_acc.columns = ["Bonnes","Total","Précision"]
    round_acc["Précision"] = round_acc["Précision"].apply(lambda x: f"{x:.1%}")
    print(round_acc.to_string())
    print(f"{'─'*70}\n")

    if errors:
        print(f"  ⚠️  Joueurs introuvables ({len(errors)}) : {errors}")

    return results_df


# ─────────────────────────────────────────────────────────────
# H. EXÉCUTION
# ─────────────────────────────────────────────────────────────

# ── 1. Prédictions sur les matchs ongoing ──
results_ongoing = predict_ongoing(show_top=30)

# ── 2. Prédictions manuelles ──
predict_match(
    "Jannik Sinner", "Carlos Alcaraz",
    match_date="2026-05-05",
    surface="Clay", tourney_level="M",
    round_str="SF", best_of=3, indoor="N"
)

predict_match(
    "Alexander Zverev", "Novak Djokovic",
    match_date="2026-05-05",
    surface="Clay", tourney_level="M",
    round_str="QF", best_of=3, indoor="N"
)

# ── 3. Impact de la surface ──
print("\n📊 Impact de la surface — Sinner vs Alcaraz :")
for surf in ["Hard", "Clay", "Grass"]:
    r = predict_match(
        "Jannik Sinner", "Carlos Alcaraz"   ,
        match_date="2026-05-05",
        surface=surf, tourney_level="G",
        round_str="F", best_of=5, indoor="N"
    )
    if r:
        print(f"   {surf:5} → Sinner {r['p1_prob']:.1%} | Alcaraz {r['p2_prob']:.1%}")


Construction index joueur → feat_clean...
   → Utilisation de p1_name (colonne existante)
✅ Index construit : 34516 lignes
✅ Modèle chargé    : LightGBM
   Features         : 94
   Matchs en base   : 17,258
   Matchs ongoing   : 95

  PRÉDICTIONS — Madrid Masters (95 matchs)
Round              Joueur 1               Joueur 2 P(J1 gagne)                 Prédit                   Réel ✅/❌
 R128       Sebastian Ofner   Nikoloz Basilashvili       44.1%   Nikoloz Basilashvili        Sebastian Ofner   ❌
 R128      Adrian Mannarino           Ignacio Buse       56.8%       Adrian Mannarino           Ignacio Buse   ❌
 R128           Zizou Bergs            Marin Cilic       48.9%            Marin Cilic            Marin Cilic   ✅
 R128       Mattia Bellucci          Damir Dzumhur       60.5%        Mattia Bellucci          Damir Dzumhur   ❌
 R128           Vit Kopriva          Zhizhen Zhang       64.8%            Vit Kopriva            Vit Kopriva   ✅
 R128         Dusan Lajovic         Lorenzo So

In [3]:
# Taux d'upsets dans ce tournoi
# (le joueur moins bien classé qui gagne)
ongoing["rank_diff"] = ongoing["winner_rank"] - ongoing["loser_rank"]
n_upsets = (ongoing["rank_diff"] > 0).sum()
print(f"Upsets dans ce tournoi : {n_upsets}/{len(ongoing)} ({n_upsets/len(ongoing):.1%})")
print(f"\nDistribution rank_diff :")
print(ongoing["rank_diff"].describe())

Upsets dans ce tournoi : 34/95 (35.8%)

Distribution rank_diff :
count     95.000000
mean     -14.968421
std       58.299384
min     -179.000000
25%      -44.000000
50%      -18.000000
75%       12.500000
max      133.000000
Name: rank_diff, dtype: float64


In [4]:
# Précision du modèle sur les matchs "normaux" vs upsets
ongoing["is_upset"] = ongoing["rank_diff"] > 0  # le moins bien classé a gagné

# Recréer results_df avec le flag upset
results_ongoing["is_upset"] = None
for i, (_, match) in enumerate(ongoing.iterrows()):
    if i < len(results_ongoing):
        results_ongoing.loc[i, "is_upset"] = match["rank_diff"] > 0

print("Précision sur matchs normaux (favori gagne) :")
normal = results_ongoing[results_ongoing["is_upset"] == False]
print(f"  {normal['_correct'].mean():.1%} ({normal['_correct'].sum()}/{len(normal)})")

print("\nPrécision sur upsets (outsider gagne) :")
upsets = results_ongoing[results_ongoing["is_upset"] == True]
print(f"  {upsets['_correct'].mean():.1%} ({upsets['_correct'].sum()}/{len(upsets)})")

print("\nSi on avait toujours prédit le mieux classé :")
baseline = (ongoing["rank_diff"] < 0).mean()
print(f"  {baseline:.1%}")


Précision sur matchs normaux (favori gagne) :
  85.2% (52/61)

Précision sur upsets (outsider gagne) :
  35.3% (12/34)

Si on avait toujours prédit le mieux classé :
  64.2%


In [5]:
# Diagnostic : que récupère get_player_stats() vraiment ?
p1 = get_player_stats("Jannik Sinner")
p2 = get_player_stats("Carlos Alcaraz")

print("Sinner :")
print(f"  rank      : {p1['rank']}")
print(f"  last10_won: {p1['last10_won']}")
print(f"  last_match: {p1['_last_match_date']}")

print("\nAlcaraz :")
print(f"  rank      : {p2['rank']}")
print(f"  last10_won: {p2['last10_won']}")
print(f"  last_match: {p2['_last_match_date']}")

# Vérifier ce que voit le modèle pour un match ongoing
match = ongoing.iloc[0]
print(f"\nMatch ongoing : {match['winner_name']} vs {match['loser_name']}")
print(f"  winner_rank : {match['winner_rank']} | loser_rank : {match['loser_rank']}")

p1s = get_player_stats(match["winner_name"])
p2s = get_player_stats(match["loser_name"])
print(f"  p1 rank récupéré : {p1s['rank'] if p1s else 'None'}")
print(f"  p2 rank récupéré : {p2s['rank'] if p2s else 'None'}")

# Vérifier l'index dans feat_clean
df_check = df_orig[(df_orig['winner_name']==match['winner_name']) |
                   (df_orig['loser_name']==match['winner_name'])]
print(f"\n  Matchs connus de {match['winner_name']} : {len(df_check)}")
print(f"  Dernier index dans df_orig : {df_check.index[-1]}")
print(f"  N_MATCHES : {N_MATCHES}")
print(f"  feat_clean length : {len(feat_clean)}")

TypeError: get_player_stats() missing 1 required positional argument: 'match_date'

In [17]:
p1 = get_player_stats("Jannik Sinner")
p2 = get_player_stats("Sebastian Ofner")
print(f"Sinner  rank: {p1['rank']} (attendu: ~1-3)")
print(f"Ofner   rank: {p2['rank']} (attendu: ~83)")

Sinner  rank: 200.0 (attendu: ~1-3)
Ofner   rank: 4.0 (attendu: ~83)


In [18]:
# Diagnostic précis de la structure de feat_clean
print("=== Structure feat_clean ===")
print(f"Longueur : {len(feat_clean)}")
print(f"\n5 premières lignes (p1_rank, label, year) :")
print(feat_clean[["p1_rank","p2_rank","label","year"]].head(10).to_string())

print(f"\n5 dernières lignes label=1 (winner view) :")
winners = feat_clean[feat_clean["label"]==1]
print(f"Nb lignes label=1 : {len(winners)}")
print(winners[["p1_rank","p2_rank","label","year"]].head(5).to_string())

print(f"\n5 premières lignes label=0 (loser view) :")
losers = feat_clean[feat_clean["label"]==0]
print(f"Nb lignes label=0 : {len(losers)}")
print(losers[["p1_rank","p2_rank","label","year"]].head(5).to_string())

# Vérifier Sinner spécifiquement
print("\n=== Dernier match Sinner dans df_orig ===")
mask = (df_orig["winner_name"]=="Jannik Sinner")|(df_orig["loser_name"]=="Jannik Sinner")
last = df_orig[mask].sort_values("tourney_date").iloc[-1]
print(f"Index dans df_orig : {last.name}")
print(f"Date : {last['tourney_date'].date()}")
print(f"Winner : {last['winner_name']} | Loser : {last['loser_name']}")
is_w = last["winner_name"] == "Jannik Sinner"
print(f"Sinner est winner : {is_w}")
print(f"\nfeat_clean.iloc[{last.name}] → p1_rank = {feat_clean.iloc[last.name]['p1_rank']}")
print(f"feat_clean.iloc[{N_MATCHES + last.name}] → p1_rank = {feat_clean.iloc[N_MATCHES + last.name]['p1_rank']}")

=== Structure feat_clean ===
Longueur : 34326

5 premières lignes (p1_rank, label, year) :
   p1_rank  p2_rank  label  year
0    157.0    999.0      1  2020
1    233.0     85.0      0  2020
2      7.0     50.0      1  2020
3     50.0      7.0      0  2020
4     79.0     87.0      1  2020
5     87.0     79.0      0  2020
6     10.0    117.0      1  2020
7    117.0     10.0      0  2020
8      5.0     40.0      1  2020
9     85.0    233.0      1  2020

5 dernières lignes label=1 (winner view) :
Nb lignes label=1 : 17163
   p1_rank  p2_rank  label  year
0    157.0    999.0      1  2020
2      7.0     50.0      1  2020
4     79.0     87.0      1  2020
6     10.0    117.0      1  2020
8      5.0     40.0      1  2020

5 premières lignes label=0 (loser view) :
Nb lignes label=0 : 17163
    p1_rank  p2_rank  label  year
1     233.0     85.0      0  2020
3      50.0      7.0      0  2020
5      87.0     79.0      0  2020
7     117.0     10.0      0  2020
10     40.0      5.0      0  2020

=== 

In [22]:
p1 = get_player_stats("Jannik Sinner")
p2 = get_player_stats("Sebastian Ofner")
print(f"Sinner rank : {p1['rank']} (attendu ~1-3)")
print(f"Ofner  rank : {p2['rank']} (attendu ~83)")

Sinner rank : 2.0 (attendu ~1-3)
Ofner  rank : 86.0 (attendu ~83)


In [23]:
# Diagnostic
print("Différentiels clés Sinner vs Alcaraz :")
p1 = get_player_stats("Jannik Sinner")
p2 = get_player_stats("Carlos Alcaraz")
for key in ["rank","last10_won","last10_ace_rate","last5_win_rate_Clay"]:
    print(f"  {key:25} Sinner={p1.get(key,'N/A')} | Alcaraz={p2.get(key,'N/A')}")

Différentiels clés Sinner vs Alcaraz :
  rank                      Sinner=2.0 | Alcaraz=2.0
  last10_won                Sinner=0.9 | Alcaraz=1.0
  last10_ace_rate           Sinner=0.0561997610403196 | Alcaraz=0.0571935832267284
  last5_win_rate_Clay       Sinner=0.8 | Alcaraz=1.0


In [ ]:
# ── Test ──
p1 = get_player_stats("Jannik Sinner")
p2 = get_player_stats("Carlos Alcaraz")
print(f"\nSinner  rank={p1['rank']} | last10_won={p1['last10_won']:.2f}")
print(f"Alcaraz rank={p2['rank']} | last10_won={p2['last10_won']:.2f}")

In [2]:
# ── Test ──
p1 = get_player_stats("Jannik Sinner")
p2 = get_player_stats("Carlos Alcaraz")
p3 = get_player_stats("Sebastian Ofner")
print(f"Sinner  rank={p1['rank']} | last10_won={p1['last10_won']:.2f} | last5_clay={p1['last5_win_rate_Clay']}")
print(f"Alcaraz rank={p2['rank']} | last10_won={p2['last10_won']:.2f} | last5_clay={p2['last5_win_rate_Clay']}")
print(f"Ofner   rank={p3['rank']} | last10_won={p3['last10_won']:.2f}")

Sinner  rank=2.0 | last10_won=1.00 | last5_clay=1.0
Alcaraz rank=2.0 | last10_won=0.70 | last5_clay=0.8
Ofner   rank=86.0 | last10_won=0.00


In [3]:
# Ajouter winner_name et loser_name directement depuis df_orig
# en utilisant le tri chronologique pour aligner les deux DataFrames

df_sorted = df_orig.sort_values("tourney_date").reset_index(drop=True)
feat_sorted = feat_clean.sort_values("year").reset_index(drop=True)

# Vérification d'alignement sur les rangs
print("Vérification alignement (5 premiers matchs) :")
for i in range(5):
    w_rank_orig = df_sorted.iloc[i]["winner_rank"]
    w_rank_feat = feat_sorted.iloc[i*2]["p1_rank"]
    l_rank_orig = df_sorted.iloc[i]["loser_rank"]
    l_rank_feat = feat_sorted.iloc[i*2+1]["p1_rank"]
    print(f"  Match {i}: orig winner_rank={w_rank_orig} feat p1_rank={w_rank_feat} "
          f"| orig loser_rank={l_rank_orig} feat p1_rank={l_rank_feat}")

Vérification alignement (5 premiers matchs) :
  Match 0: orig winner_rank=157.0 feat p1_rank=157.0 | orig loser_rank=999 feat p1_rank=25.0
  Match 1: orig winner_rank=11.0 feat p1_rank=39.0 | orig loser_rank=46 feat p1_rank=37.0
  Match 2: orig winner_rank=53.0 feat p1_rank=4.0 | orig loser_rank=423 feat p1_rank=450.0
  Match 3: orig winner_rank=20.0 feat p1_rank=108.0 | orig loser_rank=42 feat p1_rank=30.0
  Match 4: orig winner_rank=32.0 feat p1_rank=6.0 | orig loser_rank=332 feat p1_rank=999.0


In [5]:
print("p1_name" in feat_clean.columns)
print(feat_clean[["p1_name","p2_name","p1_rank","label"]].head(6).to_string())

# Test direct
sinner = feat_clean[feat_clean["p1_name"] == "Jannik Sinner"]
print(f"\nOccurrences Sinner : {len(sinner)}")
print(f"Dernière ligne : rank={sinner.iloc[-1]['p1_rank']} | last10_won={sinner.iloc[-1]['p1_last10_won']}")

alcaraz = feat_clean[feat_clean["p1_name"] == "Carlos Alcaraz"]
print(f"\nOccurrences Alcaraz : {len(alcaraz)}")
print(f"Dernière ligne : rank={alcaraz.iloc[-1]['p1_rank']} | last10_won={alcaraz.iloc[-1]['p1_last10_won']}")

True
            p1_name             p2_name  p1_rank  label
0      Steve Darcis   Alexandr Cozbinov    157.0      1
1        Oscar Otte  Daniel Elahi Galan    233.0      0
2     Andrey Rublev      Aslan Karatsev      7.0      1
3    Aslan Karatsev       Andrey Rublev     50.0      0
4     Quentin Halys    Aleksandar Vukic     79.0      1
5  Aleksandar Vukic       Quentin Halys     87.0      0

Occurrences Sinner : 422
Dernière ligne : rank=8.0 | last10_won=0.8

Occurrences Alcaraz : 371
Dernière ligne : rank=2.0 | last10_won=0.9


In [8]:
sinner_rows = feat_clean[feat_clean["p1_name"] == "Jannik Sinner"]
print(sinner_rows[["p1_name","p1_rank","p1_last10_won","year"]].tail(5).to_string())

p1 = get_player_stats("Jannik Sinner")
p2 = get_player_stats("Carlos Alcaraz")
print(f"Sinner  rank={p1['rank']} | last10_won={p1['last10_won']:.2f}")
print(f"Alcaraz rank={p2['rank']} | last10_won={p2['last10_won']:.2f}")

             p1_name  p1_rank  p1_last10_won  year
34143  Jannik Sinner      2.0            1.0  2026
34184  Jannik Sinner      2.0            1.0  2026
34192  Jannik Sinner      2.0            1.0  2026
34202  Jannik Sinner      2.0            1.0  2026
34204  Jannik Sinner      2.0            1.0  2026
Sinner  rank=2.0 | last10_won=1.00
Alcaraz rank=2.0 | last10_won=0.70


In [ ]:
import pandas as pd
import numpy as np

feat = pd.read_parquet("../../data/tennis/atp_features_clean_final.parquet")

# ─────────────────────────────────────────────────────────────
# 1. Corrélation brute avec le label
# ─────────────────────────────────────────────────────────────
print("Corrélation diff_fatigue_14d avec le label :")
print(f"  {feat['diff_fatigue_14d'].corr(feat['label']):.4f}")

# ─────────────────────────────────────────────────────────────
# 2. Win rate selon le différentiel de fatigue
# ─────────────────────────────────────────────────────────────
feat["fatigue_bucket"] = pd.cut(
    feat["diff_fatigue_14d"],
    bins=[-12, -4, -2, -1, 0, 1, 2, 4, 12],
    labels=["≤-4","[-4,-2]","[-2,-1]","[-1,0]","[0,1]","[1,2]","[2,4]","≥4"]
)

wr = feat.groupby("fatigue_bucket", observed=True)["label"].agg(["mean","count"])
wr.columns = ["Win rate P1", "Nb matchs"]
wr["Win rate P1"] = wr["Win rate P1"].apply(lambda x: f"{x:.1%}")
print("\nWin rate selon diff_fatigue_14d (P1 - P2) :")
print("(négatif = P1 plus fatigué, positif = P2 plus fatigué)")
print(wr.to_string())

# ─────────────────────────────────────────────────────────────
# 3. Feature importance du modèle
# ─────────────────────────────────────────────────────────────
import joblib
model_data = joblib.load("../../models/tennis/best_model.pkl")
model      = model_data["model"]
feat_names = model_data["feature_names"]

if hasattr(model, "pipeline"):
    importances = model.pipeline.named_steps["model"].feature_importances_
else:
    importances = model.feature_importances_

imp = pd.Series(importances, index=feat_names).sort_values(ascending=False)
fatigue_rank = imp.index.tolist().index("diff_fatigue_14d") + 1
print(f"\nRang de diff_fatigue_14d dans les feature importances : #{fatigue_rank}/{len(imp)}")
print(f"Importance : {imp['diff_fatigue_14d']:.4f}")
print(f"\nTop 10 features :")
print(imp.head(10).to_string())

Corrélation diff_fatigue_14d avec le label :
  0.4443

Win rate selon diff_fatigue_14d (P1 - P2) :
(négatif = P1 plus fatigué, positif = P2 plus fatigué)
               Win rate P1  Nb matchs
fatigue_bucket                       
≤-4                  11.9%       1955
[-4,-2]              21.4%       5167
[-2,-1]              29.5%       5616
[-1,0]               50.0%       8848
[0,1]                70.5%       5616
[1,2]                77.3%       3234
[2,4]                81.8%       2972
≥4                   93.6%        916

Rang de diff_fatigue_14d dans les feature importances : #2/84
Importance : 0.2687

Top 10 features :
diff_days_since_last    0.452413
diff_fatigue_14d        0.268707
diff_rank_pts           0.108746
diff_rank               0.048643
round_num               0.023503
p1_rank                 0.018485
p2_rank                 0.017319
tourney_level_num       0.006624
best_of                 0.002806
p1_last20_won           0.002593


In [ ]:
# ============================================================
# ÉTAPE 5 : PRÉDICTION DE MATCHS ATP — VERSION FINALE COMPLÈTE
# ============================================================
#
# Modes :
#   1. predict_match()      — prédiction manuelle (2 noms + date + contexte)
#   2. predict_from_names() — prédiction automatique (2 noms uniquement)
#                             scrape le contexte depuis l'API ATP
#   3. predict_ongoing()    — prédiction sur tous les matchs ongoing
# ============================================================

import pandas as pd
import numpy as np
import joblib
import warnings
import requests
import unicodedata
import re
from datetime import datetime
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

# ─────────────────────────────────────────────────────────────
# A. CHARGEMENT DU MODÈLE ET DES DONNÉES
# ─────────────────────────────────────────────────────────────

from sklearn.pipeline import Pipeline

class PipelineWrapper:
    def __init__(self, pipeline):
        self.pipeline = pipeline
    def predict(self, X):
        return self.pipeline.predict(X)
    def predict_proba(self, X):
        return self.pipeline.predict_proba(X)

model_data = joblib.load("../../models/tennis/best_model.pkl")
model      = model_data["model"]
model_name = model_data["model_name"]

feat_df  = pd.read_csv("../../data/tennis/atp_features_clean_final.csv")
df_orig  = pd.read_csv("../../data/tennis/atp_clean.csv", parse_dates=["tourney_date"])
df_orig  = df_orig.sort_values("tourney_date").reset_index(drop=True)

feat_clean = feat_df.copy()
feat_clean = feat_clean.sort_values("year").reset_index(drop=True)

print("Construction index joueur -> feat_clean...")
USE_PLAYER_NAME = "p1_name" in feat_clean.columns
if not USE_PLAYER_NAME:
    player_names_feat = []
    for i, row_orig in df_orig.iterrows():
        player_names_feat.append(row_orig["winner_name"])
        player_names_feat.append(row_orig["loser_name"])
    feat_clean["player_name"] = player_names_feat
else:
    print("   -> Utilisation de p1_name (colonne existante)")

print(f"OK Index construit : {len(feat_clean)} lignes")

FEATURE_COLS = [c for c in feat_clean.columns
                if c not in ["label", "year", "diff_last20_won",
                             "p1_name", "p2_name", "player_name"]]

NEW_FEATURES = ["rank_ratio", "p1_momentum", "p2_momentum", "diff_momentum",
                "p1_serve_dom", "p2_serve_dom", "diff_serve_dom", "diff_bp_pressure"]

if "feature_names" in model_data:
    FEATURE_COLS = model_data["feature_names"]
else:
    for f in NEW_FEATURES:
        if f not in FEATURE_COLS:
            FEATURE_COLS.append(f)

N_MATCHES = len(df_orig)
print(f"OK Modele charge    : {model_name}")
print(f"   Features         : {len(FEATURE_COLS)}")
print(f"   Matchs en base   : {N_MATCHES:,}")

# ─────────────────────────────────────────────────────────────
# B. ENCODAGES CONTEXTUELS
# ─────────────────────────────────────────────────────────────

ROUND_ORDER = {"R128":1,"R64":2,"R32":3,"R16":4,"QF":5,"SF":6,"F":7,"RR":3,"BR":6}
LEVEL_ORDER = {"G":5,"M":4,"F":4,"A":3,"D":2,"C":1}
HAND_MAP    = {"R":1,"L":-1,"U":0}
INDOOR_MAP  = {"Y":1,"N":0,"Unknown":0}

# ─────────────────────────────────────────────────────────────
# C. SCRAPING AUTOMATIQUE DU CONTEXTE — API ATP
# ─────────────────────────────────────────────────────────────

# ============================================================
# ÉTAPE 5 : PRÉDICTION DE MATCHS ATP — VERSION FINALE COMPLÈTE
# ============================================================
#
# Modes :
#   1. predict_match()      — prédiction manuelle (2 noms + date + contexte)
#   2. predict_from_names() — prédiction automatique (2 noms uniquement)
#                             scrape le contexte depuis l'API ATP
#   3. predict_ongoing()    — prédiction sur tous les matchs ongoing
# ============================================================

import pandas as pd
import numpy as np
import joblib
import warnings
import requests
import unicodedata
import re
from datetime import datetime
warnings.filterwarnings("ignore")
warnings.filterwarnings("ignore", category=UserWarning, module="sklearn")

# ─────────────────────────────────────────────────────────────
# A. CHARGEMENT DU MODÈLE ET DES DONNÉES
# ─────────────────────────────────────────────────────────────

from sklearn.pipeline import Pipeline

class PipelineWrapper:
    def __init__(self, pipeline):
        self.pipeline = pipeline
    def predict(self, X):
        return self.pipeline.predict(X)
    def predict_proba(self, X):
        return self.pipeline.predict_proba(X)

model_data = joblib.load("../../models/tennis/best_model.pkl")
model      = model_data["model"]
model_name = model_data["model_name"]

feat_df  = pd.read_csv("../../data/tennis/atp_features_clean_final.csv")
df_orig  = pd.read_csv("../../data/tennis/atp_clean.csv", parse_dates=["tourney_date"])
df_orig  = df_orig.sort_values("tourney_date").reset_index(drop=True)

feat_clean = feat_df.copy()
feat_clean = feat_clean.sort_values("year").reset_index(drop=True)

print("Construction index joueur -> feat_clean...")
USE_PLAYER_NAME = "p1_name" in feat_clean.columns
if not USE_PLAYER_NAME:
    player_names_feat = []
    for i, row_orig in df_orig.iterrows():
        player_names_feat.append(row_orig["winner_name"])
        player_names_feat.append(row_orig["loser_name"])
    feat_clean["player_name"] = player_names_feat
else:
    print("   -> Utilisation de p1_name (colonne existante)")

print(f"OK Index construit : {len(feat_clean)} lignes")

FEATURE_COLS = [c for c in feat_clean.columns
                if c not in ["label", "year", "diff_last20_won",
                             "p1_name", "p2_name", "player_name"]]

NEW_FEATURES = ["rank_ratio", "p1_momentum", "p2_momentum", "diff_momentum",
                "p1_serve_dom", "p2_serve_dom", "diff_serve_dom", "diff_bp_pressure"]

if "feature_names" in model_data:
    FEATURE_COLS = model_data["feature_names"]
else:
    for f in NEW_FEATURES:
        if f not in FEATURE_COLS:
            FEATURE_COLS.append(f)

N_MATCHES = len(df_orig)
print(f"OK Modele charge    : {model_name}")
print(f"   Features         : {len(FEATURE_COLS)}")
print(f"   Matchs en base   : {N_MATCHES:,}")

# ─────────────────────────────────────────────────────────────
# B. ENCODAGES CONTEXTUELS
# ─────────────────────────────────────────────────────────────

ROUND_ORDER = {"R128":1,"R64":2,"R32":3,"R16":4,"QF":5,"SF":6,"F":7,"RR":3,"BR":6}
LEVEL_ORDER = {"G":5,"M":4,"F":4,"A":3,"D":2,"C":1}
HAND_MAP    = {"R":1,"L":-1,"U":0}
INDOOR_MAP  = {"Y":1,"N":0,"Unknown":0}

# ─────────────────────────────────────────────────────────────
# C. SCRAPING AUTOMATIQUE DU CONTEXTE — API ATP
# ─────────────────────────────────────────────────────────────

SURFACE_MAP_ATP = {
    "Hard": "Hard", "Clay": "Clay", "Grass": "Grass",
    "Indoor Hard": "Hard", "Carpet": "Hard",
}
LEVEL_MAP_ATP = {
    "Grand Slam": "G", "Masters 1000": "M",
    "ATP 500": "A", "ATP 250": "A",
    "ATP Finals": "F", "Challenger": "D",
}
ROUND_MAP_ATP = {
    "F":"F","SF":"SF","QF":"QF","R16":"R16","R32":"R32",
    "R64":"R64","R128":"R128","RR":"RR",
    "Final":"F","Semifinal":"SF","Semi-Final":"SF",
    "Quarterfinal":"QF","Quarter-Final":"QF",
    "Round of 16":"R16","Round of 32":"R32",
    "Round of 64":"R64","Round of 128":"R128",
    "1st Round":"R64","2nd Round":"R32",
    "3rd Round":"R16","4th Round":"R16",
}


def normalize_name_for_search(name: str) -> str:
    """Minuscules, sans accents, sans ponctuation."""
    name = unicodedata.normalize("NFD", name)
    name = "".join(c for c in name if unicodedata.category(c) != "Mn")
    name = re.sub(r"[^\w\s]", "", name.lower().strip())
    return name


def extract_last_name(name: str) -> str:
    """
    Extrait le nom de famille depuis n'importe quel format :
      "Marcos Giron"          -> "giron"
      "Giron M."              -> "giron"
      "Roberto Bautista Agut" -> "bautista agut"
      "Bautista Agut R."      -> "bautista agut"
      "Alex de Minaur"        -> "de minaur"
    """
    name_clean = normalize_name_for_search(name)
    parts      = name_clean.split()
    if not parts:
        return ""

    particles  = {"de","del","di","van","von","der","den",
                  "le","la","los","las","da","do","dos"}
    name_parts = [p for p in parts if len(p) > 2 or p in particles]

    if not name_parts:
        return parts[-1]
    if len(name_parts) == 1:
        return name_parts[0]
    if name_parts[0] in particles:
        return " ".join(name_parts)
    if len(name_parts) >= 3:
        return " ".join(name_parts[1:])
    return name_parts[-1]


def extract_initial(name: str) -> str:
    """Extrait l'initiale du prénom."""
    name_clean = normalize_name_for_search(name)
    parts      = name_clean.split()
    particles  = {"de","del","di","van","von","der","den",
                  "le","la","los","las","da","do","dos"}
    for p in parts:
        if len(p) == 1:
            return p
    for p in parts:
        if p not in particles and len(p) > 1:
            return p[0]
    return ""


def players_match(name_dataset: str, name_source: str) -> bool:
    """
    Verifie si deux noms correspondent malgre des formats differents.
    Ex: "Marcos Giron" <-> "Giron M."
    """
    last1  = extract_last_name(name_dataset)
    last2  = extract_last_name(name_source)
    init1  = extract_initial(name_dataset)
    init2  = extract_initial(name_source)

    if last1 != last2 and last1 not in last2 and last2 not in last1:
        words1 = set(normalize_name_for_search(name_dataset).split())
        words2 = set(normalize_name_for_search(name_source).split())
        if words1 and words2:
            if len(words1 & words2) / len(words1 | words2) < 0.5:
                return False
        else:
            return False

    if init1 and init2:
        return init1 == init2
    return True


def get_atp_matches_today() -> list:
    """
    Recupere les matchs ATP prevus depuis l'API interne ATP.
    Fallback sur Flashscore si ATP echoue.
    """
    headers = {
        "User-Agent"     : "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                           "AppleWebKit/537.36 (KHTML, like Gecko) "
                           "Chrome/120.0.0.0 Safari/537.36",
        "Accept"         : "application/json, text/plain, */*",
        "Accept-Language": "fr-FR,fr;q=0.9,en;q=0.8",
        "Referer"        : "https://www.atptour.com/",
    }
    today   = datetime.now().strftime("%Y-%m-%d")
    matches = []

    # Tentative 1 : API ATP
    try:
        url  = "https://www.atptour.com/en/scores/current/atp/2026/daily-schedule"
        resp = requests.get(url, headers=headers, timeout=15)
        if resp.status_code == 200:
            try:
                data = resp.json()
                matches = _parse_atp_json(data, today)
                if matches:
                    print(f"   OK API ATP : {len(matches)} matchs trouves")
                    return matches
            except Exception:
                pass
    except Exception as e:
        print(f"   Attention API ATP : {e}")

    # Tentative 2 : Flashscore
    try:
        url = "https://d.flashscore.com/x/feed/t_1_-1_1_en_1"
        headers_fs = {**headers, "x-fsign": "SW9D1eZo",
                      "Referer": "https://www.flashscore.com/"}
        resp = requests.get(url, headers=headers_fs, timeout=10)
        if resp.status_code == 200:
            matches = _parse_flashscore_feed(resp.text, today)
            if matches:
                print(f"   OK Flashscore API : {len(matches)} matchs trouves")
                return matches
    except Exception as e:
        print(f"   Attention Flashscore API : {e}")

    print(f"   Attention : Aucune source disponible")
    return matches


def _parse_atp_json(data: dict, today: str) -> list:
    matches = []
    try:
        tournaments = data.get("tournaments", data.get("data", [data]))
        if isinstance(tournaments, dict):
            tournaments = [tournaments]
        for tourn in tournaments:
            if not isinstance(tourn, dict):
                continue
            surface    = SURFACE_MAP_ATP.get(tourn.get("surface", ""), "Hard")
            level      = LEVEL_MAP_ATP.get(tourn.get("category", ""), "A")
            tourn_name = tourn.get("name", tourn.get("tournamentName", ""))
            indoor     = "Y" if tourn.get("indoor", False) else "N"
            best_of    = 5 if level == "G" else 3
            for match in tourn.get("matches", tourn.get("events", [])):
                if not isinstance(match, dict):
                    continue
                p1 = (match.get("player1", {}) or {}).get("name", "")
                p2 = (match.get("player2", {}) or {}).get("name", "")
                if not p1 or not p2:
                    p1 = match.get("homePlayer", match.get("p1", ""))
                    p2 = match.get("awayPlayer", match.get("p2", ""))
                rd = ROUND_MAP_ATP.get(match.get("round", ""), "R32")
                dt = match.get("date", match.get("scheduledDate", today))
                if isinstance(dt, str) and len(dt) >= 10:
                    dt = dt[:10]
                if p1 and p2:
                    matches.append({
                        "player1": p1, "player2": p2,
                        "tournament_name": tourn_name,
                        "match_date": dt or today,
                        "surface": surface, "tourney_level": level,
                        "round_str": rd, "best_of": best_of, "indoor": indoor,
                    })
    except Exception as e:
        print(f"   Parsing JSON ATP : {e}")
    return matches


def _parse_flashscore_feed(text: str, today: str) -> list:
    matches = []
    try:
        for chunk in text.split("~AA~"):
            if not chunk.strip():
                continue
            fields = {}
            for part in ("~AA~" + chunk).split("¬"):
                if "~" in part:
                    k, _, v = part.partition("~")
                    fields[k.strip()] = v.strip()
            p1 = fields.get("AA", "")
            p2 = fields.get("AB", "")
            if not p1 or not p2:
                continue
            surface_raw = fields.get("EW", "hard").lower()
            level_raw   = fields.get("EC", "").lower()
            round_raw   = fields.get("EV", "")
            indoor_raw  = "Y" if "indoor" in fields.get("EX", "").lower() else "N"
            surface = SURFACE_MAP_ATP.get(surface_raw.title(), "Hard")
            level   = LEVEL_MAP_ATP.get(level_raw.title(), "A")
            rd      = ROUND_MAP_ATP.get(round_raw, "R32")
            best_of = 5 if level == "G" else 3
            matches.append({
                "player1": p1, "player2": p2,
                "tournament_name": fields.get("CT", ""),
                "match_date": today, "surface": surface,
                "tourney_level": level, "round_str": rd,
                "best_of": best_of, "indoor": indoor_raw,
            })
    except Exception as e:
        print(f"   Parsing Flashscore : {e}")
    return matches


def find_match_context(player1: str, player2: str,
                       matches: list = None) -> dict:
    """
    Cherche le contexte d'un match entre player1 et player2.
    Recherche tolerante : gere "Marcos Giron" <-> "Giron M."
    """
    if matches is None:
        print("   Recuperation des matchs ATP du jour...")
        matches = get_atp_matches_today()

    for match in matches:
        n1, n2 = match["player1"], match["player2"]
        if players_match(player1, n1) and players_match(player2, n2):
            return match
        if players_match(player1, n2) and players_match(player2, n1):
            ctx = match.copy()
            ctx["player1"], ctx["player2"] = match["player2"], match["player1"]
            return ctx
    return None


# ─────────────────────────────────────────────────────────────
# D. STATS D'UN JOUEUR
# ─────────────────────────────────────────────────────────────

def get_player_stats(player_name: str, match_date: pd.Timestamp) -> dict:
    """
    Recupere les dernieres stats rolling d'un joueur AVANT une date donnee.
    La fatigue est calculee par rapport a la date reelle du match.
    """
    mask    = (df_orig["winner_name"] == player_name) | \
              (df_orig["loser_name"]  == player_name)
    matches = df_orig[mask & (df_orig["tourney_date"] < match_date)] \
                .sort_values("tourney_date")

    if len(matches) == 0:
        return None

    last          = matches.iloc[-1]
    is_winner     = last["winner_name"] == player_name
    label_val     = 1 if is_winner else 0
    expected_rank = float(last["winner_rank"] if is_winner else last["loser_rank"])

    if USE_PLAYER_NAME:
        candidates = feat_clean[feat_clean["p1_name"] == player_name]
    else:
        candidates = feat_clean[feat_clean["player_name"] == player_name]

    candidates = candidates[candidates["label"] == label_val]
    exact      = candidates[candidates["p1_rank"] == expected_rank]
    feat_row   = exact.iloc[-1] if len(exact) > 0 else candidates.iloc[-1]

    stats = {col.replace("p1_", ""): feat_row[col]
             for col in feat_clean.columns
             if col.startswith("p1_") and col not in ["p1_name"]}

    fatigue_window       = matches[matches["tourney_date"] >=
                                   match_date - pd.Timedelta(days=14)]
    stats["fatigue_14d"]     = len(fatigue_window)
    stats["days_since_last"] = (match_date - last["tourney_date"]).days
    stats["_last_match_date"] = last["tourney_date"].date()
    stats["_last_opponent"]   = last["loser_name"] if is_winner else last["winner_name"]
    stats["_last_result"]     = "Victoire" if is_winner else "Defaite"
    return stats


# ─────────────────────────────────────────────────────────────
# E. FEATURES D'UN MATCH
# ─────────────────────────────────────────────────────────────

def build_match_features(p1_stats, p2_stats, surface, tourney_level,
                          round_str, best_of, indoor, month,
                          h2h_n=0, h2h_win_rate_p1=np.nan,
                          h2h_win_rate_surf=np.nan):
    row = {}
    row["p1_rank"] = p1_stats.get("rank", 999)
    row["p1_age"]  = p1_stats.get("age",  np.nan)
    row["p1_ht"]   = p1_stats.get("ht",   np.nan)
    row["p2_rank"] = p2_stats.get("rank", 999)
    row["p2_age"]  = p2_stats.get("age",  np.nan)
    row["p2_ht"]   = p2_stats.get("ht",   np.nan)

    row["diff_rank"]     = row["p1_rank"]    - row["p2_rank"]
    row["diff_rank_pts"] = p1_stats.get("rank_pts", 0) - p2_stats.get("rank_pts", 0)
    row["diff_seed"]     = p1_stats.get("seed", 0)     - p2_stats.get("seed", 0)
    row["diff_age"]      = row["p1_age"]     - row["p2_age"]
    row["diff_ht"]       = row["p1_ht"]      - row["p2_ht"]
    row["same_hand"]     = int(p1_stats.get("hand", 0) == p2_stats.get("hand", 0))

    rolling_stats = [
        "last5_won","last10_won","last20_won",
        "last10_ace_rate","last10_df_rate","last10_fs_in_pct",
        "last10_fs_won_pct","last10_ss_won_pct","last10_bp_saved_pct",
        "last10_n_matches",
        "last5_win_rate_Hard","last5_win_rate_Clay",
        "last10_win_rate_Hard","last10_win_rate_Clay",
        "last20_win_rate_Hard","last20_win_rate_Clay",
        "fatigue_14d","days_since_last",
    ]
    for stat in rolling_stats:
        row[f"p1_{stat}"] = p1_stats.get(stat, np.nan)
        row[f"p2_{stat}"] = p2_stats.get(stat, np.nan)

    diff_stats = [
        "last5_won","last5_ace_rate","last5_df_rate","last5_fs_in_pct",
        "last5_fs_won_pct","last5_ss_won_pct","last5_bp_saved_pct",
        "last10_won","last10_ace_rate","last10_df_rate","last10_fs_in_pct",
        "last10_fs_won_pct","last10_ss_won_pct","last10_bp_saved_pct",
        "last10_n_matches",
        "last20_ace_rate","last20_df_rate","last20_fs_in_pct",
        "last20_fs_won_pct","last20_ss_won_pct","last20_bp_saved_pct",
        "last5_win_rate_Hard","last5_win_rate_Clay",
        "last10_win_rate_Hard","last10_win_rate_Clay",
        "last20_win_rate_Hard","last20_win_rate_Clay",
        "fatigue_14d","days_since_last",
    ]
    for stat in diff_stats:
        v1, v2 = p1_stats.get(stat, np.nan), p2_stats.get(stat, np.nan)
        try:
            row[f"diff_{stat}"] = (v1 - v2 if (not np.isnan(float(v1)) and
                                                not np.isnan(float(v2))) else np.nan)
        except (TypeError, ValueError):
            row[f"diff_{stat}"] = np.nan

    row["h2h_n"]             = h2h_n
    row["h2h_win_rate_p1"]   = h2h_win_rate_p1
    row["h2h_win_rate_surf"] = h2h_win_rate_surf
    row["round_num"]         = ROUND_ORDER.get(round_str, 3)
    row["tourney_level_num"] = LEVEL_ORDER.get(tourney_level, 3)
    row["indoor_enc"]        = INDOOR_MAP.get(str(indoor), 0)
    row["month"]             = month
    row["best_of"]           = best_of
    row["surf_Clay"]         = int(surface == "Clay")
    row["surf_Grass"]        = int(surface == "Grass")
    row["surf_Hard"]         = int(surface == "Hard")

    # Nouvelles features
    r1, r2 = row["p1_rank"], row["p2_rank"]
    row["rank_ratio"] = float(np.clip(r1/r2, 0.01, 100)) if r2 > 0 else np.nan
    try:
        row["p1_momentum"] = float(row.get("p1_last5_won",np.nan)) - float(row.get("p1_last20_won",np.nan))
    except (TypeError, ValueError):
        row["p1_momentum"] = np.nan
    try:
        row["p2_momentum"] = float(row.get("p2_last5_won",np.nan)) - float(row.get("p2_last20_won",np.nan))
    except (TypeError, ValueError):
        row["p2_momentum"] = np.nan
    try:
        row["diff_momentum"] = row["p1_momentum"] - row["p2_momentum"]
    except (TypeError, ValueError):
        row["diff_momentum"] = np.nan

    def serve_dom(stats):
        try:
            return (float(stats.get("last10_ace_rate",np.nan)) +
                    float(stats.get("last10_fs_won_pct",np.nan)) -
                    float(stats.get("last10_df_rate",np.nan)))
        except (TypeError, ValueError):
            return np.nan

    row["p1_serve_dom"] = serve_dom(p1_stats)
    row["p2_serve_dom"] = serve_dom(p2_stats)
    try:
        row["diff_serve_dom"] = row["p1_serve_dom"] - row["p2_serve_dom"]
    except (TypeError, ValueError):
        row["diff_serve_dom"] = np.nan
    try:
        row["diff_bp_pressure"] = (float(row.get("diff_last10_bp_saved_pct",np.nan)) -
                                   float(row.get("diff_last10_fs_won_pct",np.nan)))
    except (TypeError, ValueError):
        row["diff_bp_pressure"] = np.nan

    X = pd.DataFrame([row])
    for col in FEATURE_COLS:
        if col not in X.columns:
            X[col] = np.nan
    return X[FEATURE_COLS]


# ─────────────────────────────────────────────────────────────
# F. H2H HISTORIQUE
# ─────────────────────────────────────────────────────────────

def get_h2h(p1_name, p2_name, surface=None):
    mask = (
        ((df_orig["winner_name"]==p1_name) & (df_orig["loser_name"]==p2_name)) |
        ((df_orig["winner_name"]==p2_name) & (df_orig["loser_name"]==p1_name))
    )
    h2h = df_orig[mask]
    n   = len(h2h)
    if n == 0:
        return 0, np.nan, np.nan
    p1_wins    = (h2h["winner_name"] == p1_name).sum()
    win_rate   = p1_wins / n
    if surface:
        h2h_s      = h2h[h2h["surface"] == surface]
        n_s        = len(h2h_s)
        p1_wins_s  = (h2h_s["winner_name"] == p1_name).sum()
        win_rate_s = p1_wins_s / n_s if n_s > 0 else np.nan
    else:
        win_rate_s = np.nan
    return n, win_rate, win_rate_s


# ─────────────────────────────────────────────────────────────
# G. PREDICTION SIMPLE — deux joueurs + date + contexte
# ─────────────────────────────────────────────────────────────

def predict_match(player1, player2, match_date,
                  surface="Hard", tourney_level="M",
                  round_str="R32", best_of=3, indoor="N"):
    """
    Predit le vainqueur d'un match.
    match_date : "YYYY-MM-DD"
    """
    date  = pd.Timestamp(match_date)
    month = date.month

    p1_stats = get_player_stats(player1, date)
    p2_stats = get_player_stats(player2, date)

    if p1_stats is None:
        print(f"Joueur introuvable : {player1}")
        return None
    if p2_stats is None:
        print(f"Joueur introuvable : {player2}")
        return None

    h2h_n, h2h_wr, h2h_wr_s = get_h2h(player1, player2, surface)
    X = build_match_features(
        p1_stats, p2_stats,
        surface=surface, tourney_level=tourney_level,
        round_str=round_str, best_of=best_of,
        indoor=indoor, month=month,
        h2h_n=h2h_n, h2h_win_rate_p1=h2h_wr, h2h_win_rate_surf=h2h_wr_s
    )

    proba       = model.predict_proba(X)[0]
    p1_win_prob = proba[1]
    p2_win_prob = proba[0]
    winner      = player1 if p1_win_prob > 0.5 else player2
    confidence  = max(p1_win_prob, p2_win_prob)

    bar_len = 40
    p1_bar  = int(p1_win_prob * bar_len)

    print(f"\n{'='*55}")
    print(f"  {player1}")
    print(f"     vs")
    print(f"  {player2}")
    print(f"  {surface} | {tourney_level} | {round_str} | Best of {best_of} | {match_date}")
    print(f"{'='*55}")
    print(f"\n  Derniers matchs connus :")
    print(f"  {player1[:25]:25} -> {p1_stats['_last_match_date']} "
          f"({p1_stats['_last_result']} vs {p1_stats['_last_opponent'][:20]})")
    print(f"  {player2[:25]:25} -> {p2_stats['_last_match_date']} "
          f"({p2_stats['_last_result']} vs {p2_stats['_last_opponent'][:20]})")
    print(f"\n  Fatigue (14j avant le match) :")
    print(f"  {player1[:25]:25} -> {int(p1_stats['fatigue_14d'])} matchs | "
          f"repos : {int(p1_stats['days_since_last'])} jours")
    print(f"  {player2[:25]:25} -> {int(p2_stats['fatigue_14d'])} matchs | "
          f"repos : {int(p2_stats['days_since_last'])} jours")
    if h2h_n > 0:
        p1_h2h_wins = int(h2h_wr * h2h_n)
        print(f"\n  H2H : {player1} mene {p1_h2h_wins}-{h2h_n-p1_h2h_wins} "
              f"({h2h_wr:.0%} de victoires)")
        if not np.isnan(h2h_wr_s):
            print(f"        Sur {surface} : {h2h_wr_s:.0%}")
    else:
        print(f"\n  H2H : Aucun affrontement precedent")
    print(f"\n  |{'='*p1_bar}{'-'*(bar_len-p1_bar)}|")
    print(f"  {player1[:22]:22} {p1_win_prob:.1%}  |  {p2_win_prob:.1%} {player2[:22]}")
    print(f"\n  Vainqueur predit : {winner}")
    print(f"  Confiance        : {confidence:.1%}")
    print(f"  Modele           : {model_name}")
    print(f"{'='*55}\n")

    return {"player1": player1, "player2": player2,
            "p1_prob": p1_win_prob, "p2_prob": p2_win_prob,
            "predicted_winner": winner, "confidence": confidence}


# ─────────────────────────────────────────────────────────────
# H. PREDICTION AUTOMATIQUE — juste les deux noms
# ─────────────────────────────────────────────────────────────

def predict_from_names(player1: str, player2: str) -> dict:
    """
    Predit le resultat d'un match en recuperant automatiquement
    le contexte depuis l'API ATP / Flashscore.

    Si le match n'est pas trouve, demande les infos manuellement.

    Exemple :
        predict_from_names("Jannik Sinner", "Carlos Alcaraz")
    """
    print(f"\nRecherche : {player1} vs {player2}")
    print("-" * 50)

    context = find_match_context(player1, player2)

    if context is None:
        print("Match non trouve automatiquement.")
        print("Entrez les informations manuellement :\n")

        today      = datetime.now().strftime("%Y-%m-%d")
        match_date = input(f"   Date du match [{today}] : ").strip() or today

        surface_in = input("   Surface (Hard/Clay/Grass) [Hard] : ").strip()
        surface    = surface_in if surface_in in ["Hard","Clay","Grass"] else "Hard"

        level_in   = input("   Niveau (G/M/A/D/F/C) [M] : ").strip().upper()
        level      = level_in if level_in in ["G","M","A","D","F","C"] else "M"

        round_in   = input("   Tour (R128/R64/R32/R16/QF/SF/F) [R32] : ").strip()
        round_str  = round_in if round_in in ROUND_ORDER else "R32"

        bo_in      = input("   Best of (3 ou 5) [3] : ").strip()
        best_of    = 5 if bo_in == "5" else 3

        indoor_in  = input("   Indoor ? (Y/N) [N] : ").strip().upper()
        indoor     = "Y" if indoor_in == "Y" else "N"

        context = {"match_date": match_date, "surface": surface,
                   "tourney_level": level, "round_str": round_str,
                   "best_of": best_of, "indoor": indoor,
                   "tournament_name": "Inconnu"}

    print(f"\nContexte trouve :")
    print(f"   Tournoi : {context.get('tournament_name','N/A')}")
    print(f"   Date    : {context['match_date']}")
    print(f"   Surface : {context['surface']} | "
          f"Niveau : {context['tourney_level']} | "
          f"Tour : {context['round_str']} | "
          f"Best of {context['best_of']}")

    return predict_match(
        player1, player2,
        match_date    = context["match_date"],
        surface       = context["surface"],
        tourney_level = context["tourney_level"],
        round_str     = context["round_str"],
        best_of       = context["best_of"],
        indoor        = context["indoor"],
    )


# ─────────────────────────────────────────────────────────────
# I. PREDICTION SUR LES MATCHS ONGOING
# ─────────────────────────────────────────────────────────────

def predict_ongoing(ongoing_df: pd.DataFrame = None, show_top: int = 30):
    """
    Predit le vainqueur de tous les matchs d'un fichier ongoing.
    Utilise la date reelle de chaque match pour la fatigue.
    """
    import random
    random.seed(42)

    if ongoing_df is None:
        try:
            ongoing_df = pd.read_csv("../../data/tennis/ongoing_tourneys.csv")
        except FileNotFoundError:
            print("Fichier ongoing_tourneys.csv introuvable")
            return None

    print(f"\n{'='*70}")
    print(f"  PREDICTIONS -- {ongoing_df['tourney_name'].iloc[0]} "
          f"({len(ongoing_df)} matchs)")
    print(f"{'='*70}")

    preds, errors = [], []

    for _, match in ongoing_df.iterrows():
        match_date = pd.to_datetime(str(match["tourney_date"]), format="%Y%m%d")
        month      = match_date.month

        if random.random() > 0.5:
            p1_name, p2_name = match["winner_name"], match["loser_name"]
        else:
            p1_name, p2_name = match["loser_name"], match["winner_name"]

        real_winner = match["winner_name"]
        surface     = match["surface"]
        level       = match["tourney_level"]
        round_s     = match["round"]
        best_of     = match["best_of"]
        indoor      = match.get("indoor", "N")

        p1_stats = get_player_stats(p1_name, match_date)
        p2_stats = get_player_stats(p2_name, match_date)

        if p1_stats is None or p2_stats is None:
            errors.append(p1_name if p1_stats is None else p2_name)
            continue

        h2h_n, h2h_wr, h2h_wr_s = get_h2h(p1_name, p2_name, surface)
        if p1_name == match["loser_name"] and not np.isnan(h2h_wr):
            h2h_wr   = 1 - h2h_wr
            h2h_wr_s = 1 - h2h_wr_s if not np.isnan(h2h_wr_s) else np.nan

        X = build_match_features(
            p1_stats, p2_stats,
            surface=surface, tourney_level=level,
            round_str=round_s, best_of=best_of,
            indoor=str(indoor), month=month,
            h2h_n=h2h_n, h2h_win_rate_p1=h2h_wr, h2h_win_rate_surf=h2h_wr_s
        )

        proba       = model.predict_proba(X)[0]
        p1_prob     = proba[1]
        pred_winner = p1_name if p1_prob > 0.5 else p2_name
        correct     = pred_winner == real_winner

        preds.append({
            "Round": round_s, "Joueur 1": p1_name, "Joueur 2": p2_name,
            "P(J1 gagne)": f"{p1_prob:.1%}", "Predit": pred_winner,
            "Reel": real_winner, "OK/KO": "OK" if correct else "KO",
            "_correct": correct,
        })

    results_df = pd.DataFrame(preds)
    accuracy   = results_df["_correct"].mean()
    n_correct  = results_df["_correct"].sum()

    display_cols = ["Round","Joueur 1","Joueur 2","P(J1 gagne)","Predit","Reel","OK/KO"]
    print(results_df[display_cols].head(show_top).to_string(index=False))
    if len(results_df) > show_top:
        print(f"  ... ({len(results_df) - show_top} matchs supplementaires)")
    print(f"\n{'-'*70}")
    print(f"  {len(results_df)} matchs : OK {n_correct} ({accuracy:.1%}) "
          f"| KO {len(results_df)-n_correct} ({1-accuracy:.1%})")
    round_acc = results_df.groupby("Round")["_correct"].agg(["sum","count","mean"])
    round_acc.columns = ["Bonnes","Total","Precision"]
    round_acc["Precision"] = round_acc["Precision"].apply(lambda x: f"{x:.1%}")
    print(f"\n  Precision par round :\n{round_acc.to_string()}")
    print(f"{'-'*70}\n")
    if errors:
        print(f"  Joueurs introuvables ({len(errors)}) : {errors}")
    return results_df


# ─────────────────────────────────────────────────────────────
# J. EXECUTION
# ─────────────────────────────────────────────────────────────

# Mode 1 : prediction automatique (juste les noms)
predict_from_names("Jannik Sinner", "Carlos Alcaraz")

# Mode 2 : prediction manuelle (noms + contexte complet)
predict_match(
    "Alexander Zverev", "Novak Djokovic",
    match_date="2026-05-05",
    surface="Clay", tourney_level="M",
    round_str="QF", best_of=3, indoor="N"
)

# Mode 3 : prediction sur les matchs ongoing
# predict_ongoing()


# ─────────────────────────────────────────────────────────────
# D. STATS D'UN JOUEUR
# ─────────────────────────────────────────────────────────────

def get_player_stats(player_name: str, match_date: pd.Timestamp) -> dict:
    """
    Recupere les dernieres stats rolling d'un joueur AVANT une date donnee.
    La fatigue est calculee par rapport a la date reelle du match.
    """
    mask    = (df_orig["winner_name"] == player_name) | \
              (df_orig["loser_name"]  == player_name)
    matches = df_orig[mask & (df_orig["tourney_date"] < match_date)] \
                .sort_values("tourney_date")

    if len(matches) == 0:
        return None

    last          = matches.iloc[-1]
    is_winner     = last["winner_name"] == player_name
    label_val     = 1 if is_winner else 0
    expected_rank = float(last["winner_rank"] if is_winner else last["loser_rank"])

    if USE_PLAYER_NAME:
        candidates = feat_clean[feat_clean["p1_name"] == player_name]
    else:
        candidates = feat_clean[feat_clean["player_name"] == player_name]

    candidates = candidates[candidates["label"] == label_val]
    exact      = candidates[candidates["p1_rank"] == expected_rank]
    feat_row   = exact.iloc[-1] if len(exact) > 0 else candidates.iloc[-1]

    stats = {col.replace("p1_", ""): feat_row[col]
             for col in feat_clean.columns
             if col.startswith("p1_") and col not in ["p1_name"]}

    fatigue_window       = matches[matches["tourney_date"] >=
                                   match_date - pd.Timedelta(days=14)]
    stats["fatigue_14d"]     = len(fatigue_window)
    stats["days_since_last"] = (match_date - last["tourney_date"]).days
    stats["_last_match_date"] = last["tourney_date"].date()
    stats["_last_opponent"]   = last["loser_name"] if is_winner else last["winner_name"]
    stats["_last_result"]     = "Victoire" if is_winner else "Defaite"
    return stats


# ─────────────────────────────────────────────────────────────
# E. FEATURES D'UN MATCH
# ─────────────────────────────────────────────────────────────

def build_match_features(p1_stats, p2_stats, surface, tourney_level,
                          round_str, best_of, indoor, month,
                          h2h_n=0, h2h_win_rate_p1=np.nan,
                          h2h_win_rate_surf=np.nan):
    row = {}
    row["p1_rank"] = p1_stats.get("rank", 999)
    row["p1_age"]  = p1_stats.get("age",  np.nan)
    row["p1_ht"]   = p1_stats.get("ht",   np.nan)
    row["p2_rank"] = p2_stats.get("rank", 999)
    row["p2_age"]  = p2_stats.get("age",  np.nan)
    row["p2_ht"]   = p2_stats.get("ht",   np.nan)

    row["diff_rank"]     = row["p1_rank"]    - row["p2_rank"]
    row["diff_rank_pts"] = p1_stats.get("rank_pts", 0) - p2_stats.get("rank_pts", 0)
    row["diff_seed"]     = p1_stats.get("seed", 0)     - p2_stats.get("seed", 0)
    row["diff_age"]      = row["p1_age"]     - row["p2_age"]
    row["diff_ht"]       = row["p1_ht"]      - row["p2_ht"]
    row["same_hand"]     = int(p1_stats.get("hand", 0) == p2_stats.get("hand", 0))

    rolling_stats = [
        "last5_won","last10_won","last20_won",
        "last10_ace_rate","last10_df_rate","last10_fs_in_pct",
        "last10_fs_won_pct","last10_ss_won_pct","last10_bp_saved_pct",
        "last10_n_matches",
        "last5_win_rate_Hard","last5_win_rate_Clay",
        "last10_win_rate_Hard","last10_win_rate_Clay",
        "last20_win_rate_Hard","last20_win_rate_Clay",
        "fatigue_14d","days_since_last",
    ]
    for stat in rolling_stats:
        row[f"p1_{stat}"] = p1_stats.get(stat, np.nan)
        row[f"p2_{stat}"] = p2_stats.get(stat, np.nan)

    diff_stats = [
        "last5_won","last5_ace_rate","last5_df_rate","last5_fs_in_pct",
        "last5_fs_won_pct","last5_ss_won_pct","last5_bp_saved_pct",
        "last10_won","last10_ace_rate","last10_df_rate","last10_fs_in_pct",
        "last10_fs_won_pct","last10_ss_won_pct","last10_bp_saved_pct",
        "last10_n_matches",
        "last20_ace_rate","last20_df_rate","last20_fs_in_pct",
        "last20_fs_won_pct","last20_ss_won_pct","last20_bp_saved_pct",
        "last5_win_rate_Hard","last5_win_rate_Clay",
        "last10_win_rate_Hard","last10_win_rate_Clay",
        "last20_win_rate_Hard","last20_win_rate_Clay",
        "fatigue_14d","days_since_last",
    ]
    for stat in diff_stats:
        v1, v2 = p1_stats.get(stat, np.nan), p2_stats.get(stat, np.nan)
        try:
            row[f"diff_{stat}"] = (v1 - v2 if (not np.isnan(float(v1)) and
                                                not np.isnan(float(v2))) else np.nan)
        except (TypeError, ValueError):
            row[f"diff_{stat}"] = np.nan

    row["h2h_n"]             = h2h_n
    row["h2h_win_rate_p1"]   = h2h_win_rate_p1
    row["h2h_win_rate_surf"] = h2h_win_rate_surf
    row["round_num"]         = ROUND_ORDER.get(round_str, 3)
    row["tourney_level_num"] = LEVEL_ORDER.get(tourney_level, 3)
    row["indoor_enc"]        = INDOOR_MAP.get(str(indoor), 0)
    row["month"]             = month
    row["best_of"]           = best_of
    row["surf_Clay"]         = int(surface == "Clay")
    row["surf_Grass"]        = int(surface == "Grass")
    row["surf_Hard"]         = int(surface == "Hard")

    # Nouvelles features
    r1, r2 = row["p1_rank"], row["p2_rank"]
    row["rank_ratio"] = float(np.clip(r1/r2, 0.01, 100)) if r2 > 0 else np.nan
    try:
        row["p1_momentum"] = float(row.get("p1_last5_won",np.nan)) - float(row.get("p1_last20_won",np.nan))
    except (TypeError, ValueError):
        row["p1_momentum"] = np.nan
    try:
        row["p2_momentum"] = float(row.get("p2_last5_won",np.nan)) - float(row.get("p2_last20_won",np.nan))
    except (TypeError, ValueError):
        row["p2_momentum"] = np.nan
    try:
        row["diff_momentum"] = row["p1_momentum"] - row["p2_momentum"]
    except (TypeError, ValueError):
        row["diff_momentum"] = np.nan

    def serve_dom(stats):
        try:
            return (float(stats.get("last10_ace_rate",np.nan)) +
                    float(stats.get("last10_fs_won_pct",np.nan)) -
                    float(stats.get("last10_df_rate",np.nan)))
        except (TypeError, ValueError):
            return np.nan

    row["p1_serve_dom"] = serve_dom(p1_stats)
    row["p2_serve_dom"] = serve_dom(p2_stats)
    try:
        row["diff_serve_dom"] = row["p1_serve_dom"] - row["p2_serve_dom"]
    except (TypeError, ValueError):
        row["diff_serve_dom"] = np.nan
    try:
        row["diff_bp_pressure"] = (float(row.get("diff_last10_bp_saved_pct",np.nan)) -
                                   float(row.get("diff_last10_fs_won_pct",np.nan)))
    except (TypeError, ValueError):
        row["diff_bp_pressure"] = np.nan

    X = pd.DataFrame([row])
    for col in FEATURE_COLS:
        if col not in X.columns:
            X[col] = np.nan
    return X[FEATURE_COLS]


# ─────────────────────────────────────────────────────────────
# F. H2H HISTORIQUE
# ─────────────────────────────────────────────────────────────

def get_h2h(p1_name, p2_name, surface=None):
    mask = (
        ((df_orig["winner_name"]==p1_name) & (df_orig["loser_name"]==p2_name)) |
        ((df_orig["winner_name"]==p2_name) & (df_orig["loser_name"]==p1_name))
    )
    h2h = df_orig[mask]
    n   = len(h2h)
    if n == 0:
        return 0, np.nan, np.nan
    p1_wins    = (h2h["winner_name"] == p1_name).sum()
    win_rate   = p1_wins / n
    if surface:
        h2h_s      = h2h[h2h["surface"] == surface]
        n_s        = len(h2h_s)
        p1_wins_s  = (h2h_s["winner_name"] == p1_name).sum()
        win_rate_s = p1_wins_s / n_s if n_s > 0 else np.nan
    else:
        win_rate_s = np.nan
    return n, win_rate, win_rate_s


# ─────────────────────────────────────────────────────────────
# G. PREDICTION SIMPLE — deux joueurs + date + contexte
# ─────────────────────────────────────────────────────────────

def predict_match(player1, player2, match_date,
                  surface="Hard", tourney_level="M",
                  round_str="R32", best_of=3, indoor="N"):
    """
    Predit le vainqueur d'un match.
    match_date : "YYYY-MM-DD"
    """
    date  = pd.Timestamp(match_date)
    month = date.month

    p1_stats = get_player_stats(player1, date)
    p2_stats = get_player_stats(player2, date)

    if p1_stats is None:
        print(f"Joueur introuvable : {player1}")
        return None
    if p2_stats is None:
        print(f"Joueur introuvable : {player2}")
        return None

    h2h_n, h2h_wr, h2h_wr_s = get_h2h(player1, player2, surface)
    X = build_match_features(
        p1_stats, p2_stats,
        surface=surface, tourney_level=tourney_level,
        round_str=round_str, best_of=best_of,
        indoor=indoor, month=month,
        h2h_n=h2h_n, h2h_win_rate_p1=h2h_wr, h2h_win_rate_surf=h2h_wr_s
    )

    proba       = model.predict_proba(X)[0]
    p1_win_prob = proba[1]
    p2_win_prob = proba[0]
    winner      = player1 if p1_win_prob > 0.5 else player2
    confidence  = max(p1_win_prob, p2_win_prob)

    bar_len = 40
    p1_bar  = int(p1_win_prob * bar_len)

    print(f"\n{'='*55}")
    print(f"  {player1}")
    print(f"     vs")
    print(f"  {player2}")
    print(f"  {surface} | {tourney_level} | {round_str} | Best of {best_of} | {match_date}")
    print(f"{'='*55}")
    print(f"\n  Derniers matchs connus :")
    print(f"  {player1[:25]:25} -> {p1_stats['_last_match_date']} "
          f"({p1_stats['_last_result']} vs {p1_stats['_last_opponent'][:20]})")
    print(f"  {player2[:25]:25} -> {p2_stats['_last_match_date']} "
          f"({p2_stats['_last_result']} vs {p2_stats['_last_opponent'][:20]})")
    print(f"\n  Fatigue (14j avant le match) :")
    print(f"  {player1[:25]:25} -> {int(p1_stats['fatigue_14d'])} matchs | "
          f"repos : {int(p1_stats['days_since_last'])} jours")
    print(f"  {player2[:25]:25} -> {int(p2_stats['fatigue_14d'])} matchs | "
          f"repos : {int(p2_stats['days_since_last'])} jours")
    if h2h_n > 0:
        p1_h2h_wins = int(h2h_wr * h2h_n)
        print(f"\n  H2H : {player1} mene {p1_h2h_wins}-{h2h_n-p1_h2h_wins} "
              f"({h2h_wr:.0%} de victoires)")
        if not np.isnan(h2h_wr_s):
            print(f"        Sur {surface} : {h2h_wr_s:.0%}")
    else:
        print(f"\n  H2H : Aucun affrontement precedent")
    print(f"\n  |{'='*p1_bar}{'-'*(bar_len-p1_bar)}|")
    print(f"  {player1[:22]:22} {p1_win_prob:.1%}  |  {p2_win_prob:.1%} {player2[:22]}")
    print(f"\n  Vainqueur predit : {winner}")
    print(f"  Confiance        : {confidence:.1%}")
    print(f"  Modele           : {model_name}")
    print(f"{'='*55}\n")

    return {"player1": player1, "player2": player2,
            "p1_prob": p1_win_prob, "p2_prob": p2_win_prob,
            "predicted_winner": winner, "confidence": confidence}


# ─────────────────────────────────────────────────────────────
# H. PREDICTION AUTOMATIQUE — juste les deux noms
# ─────────────────────────────────────────────────────────────

def predict_from_names(player1: str, player2: str) -> dict:
    """
    Predit le resultat d'un match en recuperant automatiquement
    le contexte depuis l'API ATP / Flashscore.

    Si le match n'est pas trouve, demande les infos manuellement.

    Exemple :
        predict_from_names("Jannik Sinner", "Carlos Alcaraz")
    """
    print(f"\nRecherche : {player1} vs {player2}")
    print("-" * 50)

    context = find_match_context(player1, player2)

    if context is None:
        print("Match non trouve automatiquement.")
        print("Entrez les informations manuellement :\n")

        today      = datetime.now().strftime("%Y-%m-%d")
        match_date = input(f"   Date du match [{today}] : ").strip() or today

        surface_in = input("   Surface (Hard/Clay/Grass) [Hard] : ").strip()
        surface    = surface_in if surface_in in ["Hard","Clay","Grass"] else "Hard"

        level_in   = input("   Niveau (G/M/A/D/F/C) [M] : ").strip().upper()
        level      = level_in if level_in in ["G","M","A","D","F","C"] else "M"

        round_in   = input("   Tour (R128/R64/R32/R16/QF/SF/F) [R32] : ").strip()
        round_str  = round_in if round_in in ROUND_ORDER else "R32"

        bo_in      = input("   Best of (3 ou 5) [3] : ").strip()
        best_of    = 5 if bo_in == "5" else 3

        indoor_in  = input("   Indoor ? (Y/N) [N] : ").strip().upper()
        indoor     = "Y" if indoor_in == "Y" else "N"

        context = {"match_date": match_date, "surface": surface,
                   "tourney_level": level, "round_str": round_str,
                   "best_of": best_of, "indoor": indoor,
                   "tournament_name": "Inconnu"}

    print(f"\nContexte trouve :")
    print(f"   Tournoi : {context.get('tournament_name','N/A')}")
    print(f"   Date    : {context['match_date']}")
    print(f"   Surface : {context['surface']} | "
          f"Niveau : {context['tourney_level']} | "
          f"Tour : {context['round_str']} | "
          f"Best of {context['best_of']}")

    return predict_match(
        player1, player2,
        match_date    = context["match_date"],
        surface       = context["surface"],
        tourney_level = context["tourney_level"],
        round_str     = context["round_str"],
        best_of       = context["best_of"],
        indoor        = context["indoor"],
    )


# ─────────────────────────────────────────────────────────────
# I. PREDICTION SUR LES MATCHS ONGOING
# ─────────────────────────────────────────────────────────────

def predict_ongoing(ongoing_df: pd.DataFrame = None, show_top: int = 30):
    """
    Predit le vainqueur de tous les matchs d'un fichier ongoing.
    Utilise la date reelle de chaque match pour la fatigue.
    """
    import random
    random.seed(42)

    if ongoing_df is None:
        try:
            ongoing_df = pd.read_csv("../../data/tennis/ongoing_tourneys.csv")
        except FileNotFoundError:
            print("Fichier ongoing_tourneys.csv introuvable")
            return None

    print(f"\n{'='*70}")
    print(f"  PREDICTIONS -- {ongoing_df['tourney_name'].iloc[0]} "
          f"({len(ongoing_df)} matchs)")
    print(f"{'='*70}")

    preds, errors = [], []

    for _, match in ongoing_df.iterrows():
        match_date = pd.to_datetime(str(match["tourney_date"]), format="%Y%m%d")
        month      = match_date.month

        if random.random() > 0.5:
            p1_name, p2_name = match["winner_name"], match["loser_name"]
        else:
            p1_name, p2_name = match["loser_name"], match["winner_name"]

        real_winner = match["winner_name"]
        surface     = match["surface"]
        level       = match["tourney_level"]
        round_s     = match["round"]
        best_of     = match["best_of"]
        indoor      = match.get("indoor", "N")

        p1_stats = get_player_stats(p1_name, match_date)
        p2_stats = get_player_stats(p2_name, match_date)

        if p1_stats is None or p2_stats is None:
            errors.append(p1_name if p1_stats is None else p2_name)
            continue

        h2h_n, h2h_wr, h2h_wr_s = get_h2h(p1_name, p2_name, surface)
        if p1_name == match["loser_name"] and not np.isnan(h2h_wr):
            h2h_wr   = 1 - h2h_wr
            h2h_wr_s = 1 - h2h_wr_s if not np.isnan(h2h_wr_s) else np.nan

        X = build_match_features(
            p1_stats, p2_stats,
            surface=surface, tourney_level=level,
            round_str=round_s, best_of=best_of,
            indoor=str(indoor), month=month,
            h2h_n=h2h_n, h2h_win_rate_p1=h2h_wr, h2h_win_rate_surf=h2h_wr_s
        )

        proba       = model.predict_proba(X)[0]
        p1_prob     = proba[1]
        pred_winner = p1_name if p1_prob > 0.5 else p2_name
        correct     = pred_winner == real_winner

        preds.append({
            "Round": round_s, "Joueur 1": p1_name, "Joueur 2": p2_name,
            "P(J1 gagne)": f"{p1_prob:.1%}", "Predit": pred_winner,
            "Reel": real_winner, "OK/KO": "OK" if correct else "KO",
            "_correct": correct,
        })

    results_df = pd.DataFrame(preds)
    accuracy   = results_df["_correct"].mean()
    n_correct  = results_df["_correct"].sum()

    display_cols = ["Round","Joueur 1","Joueur 2","P(J1 gagne)","Predit","Reel","OK/KO"]
    print(results_df[display_cols].head(show_top).to_string(index=False))
    if len(results_df) > show_top:
        print(f"  ... ({len(results_df) - show_top} matchs supplementaires)")
    print(f"\n{'-'*70}")
    print(f"  {len(results_df)} matchs : OK {n_correct} ({accuracy:.1%}) "
          f"| KO {len(results_df)-n_correct} ({1-accuracy:.1%})")
    round_acc = results_df.groupby("Round")["_correct"].agg(["sum","count","mean"])
    round_acc.columns = ["Bonnes","Total","Precision"]
    round_acc["Precision"] = round_acc["Precision"].apply(lambda x: f"{x:.1%}")
    print(f"\n  Precision par round :\n{round_acc.to_string()}")
    print(f"{'-'*70}\n")
    if errors:
        print(f"  Joueurs introuvables ({len(errors)}) : {errors}")
    return results_df


# ─────────────────────────────────────────────────────────────
# J. EXECUTION
# ─────────────────────────────────────────────────────────────

# Mode 1 : prediction automatique (juste les noms)
predict_from_names("Matteo Arnaldi", "Jaume Munar")

# Mode 2 : prediction manuelle (noms + contexte complet)
predict_match(
    "Alexander Zverev", "Novak Djokovic",
    match_date="2026-05-05",
    surface="Clay", tourney_level="M",
    round_str="QF", best_of=3, indoor="N"
)

# Mode 3 : prediction sur les matchs ongoing
# predict_ongoing()

Construction index joueur -> feat_clean...
   -> Utilisation de p1_name (colonne existante)
OK Index construit : 34326 lignes
OK Modele charge    : LightGBM
   Features         : 94
   Matchs en base   : 17,163
Construction index joueur -> feat_clean...
   -> Utilisation de p1_name (colonne existante)
OK Index construit : 34326 lignes
OK Modele charge    : LightGBM
   Features         : 94
   Matchs en base   : 17,163

Recherche : Jannik Sinner vs Carlos Alcaraz
--------------------------------------------------
   Recuperation des matchs ATP du jour...
   Attention : Aucune source disponible
Match non trouve automatiquement.
Entrez les informations manuellement :

